In [ ]:
# Cell 1：最初はこのセルだけ実行し、入力欄を自分の値に変更してください。
# SET_ / REPLACE_WITH_ は未設定です。既存Widgetは再作成しません。
defaults = {
    "census_year": "2020",
    "source_run_id": "SET_NEW_SOURCE_RUN_ID",
    "raw_root": "/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_RAW_VOLUME",
    "work_root": "/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_WORK_VOLUME"
}
labels = {
    "census_year": "取込み年度：2020 または 2025",
    "source_run_id": "今回の年度取込みに付ける未使用ID",
    "raw_root": "原本Volumeの絶対パス（末尾に年度を付けない）",
    "work_root": "成果物Volumeの絶対パス（4本で共通）"
}
for name, value in defaults.items():
    try:
        aidputils.widgets.get(name)
    except Exception:
        aidputils.widgets.text(name=name, defaultValue=value, label=labels[name])
for name in defaults:
    print(name, '=', aidputils.widgets.get(name))


# nb01: 調査年を切り替えて原数値の62市区町村を取り込む

`census_year=2020` と `2025` を、別々の `source_run_id` で各1回実行します。
原本は raw_root/調査年/ の3ファイルです。2020と2025は表頭の行順が違うため、年別定義を使います。

データは実Excelでローカル検査済み。AIDP上の実行結果は実機で確認してください。
年齢不詳の `-` は前編互換でNULLのまま保持し、0への自動変換は行いません。


In [ ]:
import json, re
from pathlib import Path
from pyspark.sql import functions as F, types as T

for name, default in [
    ('census_year', '2020'),
    ('source_run_id', 'SET_UNIQUE_SOURCE_RUN_ID'),
    ('raw_root', '/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_RAW_VOLUME'),
    ('work_root', '/Volumes/REPLACE_WITH_CATALOG/REPLACE_WITH_SCHEMA/REPLACE_WITH_WORK_VOLUME'),
]:
    try:
        aidputils.widgets.get(name)
    except Exception:
        aidputils.widgets.text(name=name, defaultValue=default, label=name)

CENSUS_YEAR = int(aidputils.widgets.get('census_year'))
SOURCE_RUN_ID = aidputils.widgets.get('source_run_id').strip()
RAW_ROOT = aidputils.widgets.get('raw_root').rstrip('/')
WORK_ROOT = aidputils.widgets.get('work_root').rstrip('/')
if CENSUS_YEAR not in (2020, 2025):
    raise ValueError('census_year must be 2020 or 2025')
if not re.fullmatch(r'[A-Za-z0-9_-]{1,100}', SOURCE_RUN_ID) or SOURCE_RUN_ID.startswith('SET_'):
    raise ValueError('Set a unique source_run_id, e.g. census2020_my01')
for root in [RAW_ROOT, WORK_ROOT]:
    if 'REPLACE_WITH_' in root or not root.startswith('/Volumes/') or '..' in Path(root).parts:
        raise ValueError('Use an absolute /Volumes/... root without ..')
RUN_ROOT = f'{WORK_ROOT}/runs/{SOURCE_RUN_ID}'
TARGET = f'{RUN_ROOT}/curated/tokyo_year'
print({'census_year': CENSUS_YEAR, 'source_run_id': SOURCE_RUN_ID, 'target': TARGET})


## 年別の読込み定義

以下は同梱 `config/source_registry.json` と同じ内容です。外部ファイル読込みを必要としないようNotebookに埋め込んでいます。
2020の項目名行と2025の項目名行は同じ位置ではありません。表頭はセル単位で検査します。
SHA-256が変わった原本は、公式の改訂情報と表頭・総数を再確認して定義を更新するまで停止します。


In [ ]:
SOURCE_REGISTRY = json.loads(r'''
{
  "2020": {
    "census_year": 2020,
    "reference_date": "2020-10-01",
    "population_previous_year": 2015,
    "series": "original",
    "boundary_basis": "survey_date",
    "unknown_policy": "preserve_dash_as_null",
    "sources": {
      "population": {
        "filename": "population_1_1.xlsx",
        "sheet": "b01_01",
        "first_data_row": 16,
        "stat_infid": "000032142402",
        "page_url": "https://www.e-stat.go.jp/stat-search/files?stat_infid=000032142402",
        "download_url": "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032142402&fileKind=0",
        "columns": {
          "region_type": "A",
          "prefecture": "E",
          "area_code": "F",
          "area_name": "G",
          "population": "H",
          "population_previous_rearranged": "K",
          "population_change_pct_source": "M",
          "households": "Q",
          "general_households": "R"
        },
        "header_anchors": {
          "A1": "令和２年国勢調査　人口等基本集計",
          "A2": "第１－１表　男女別人口，世帯の種類別世帯数及び世帯人員並びに2015年（平成27年）の人口（組替），2015年（平成27年）の世帯数（組替），5年間の人口増減数，5年間の人口増減率，5年間の世帯増減数，5年間の世帯増減率，人口性比，面積（参考）及び人口密度－全国，都道府県，市区町村（2000年（平成12年）市区町村含む）",
          "A15": "地域識別コード",
          "E15": "2020年_都道府県",
          "F15": "2020年_地域コード",
          "G15": "地域名",
          "H10": "人口",
          "H13": "0_総数",
          "H14": "人",
          "K10": "2015年（平成27年）の人口（組替）",
          "K14": "人",
          "M10": "5年間の人口増減率",
          "M14": "％",
          "Q10": "世帯数",
          "R10": "世帯数",
          "R13": "1_一般世帯",
          "R14": "世帯"
        },
        "verified_file_sha256": "91ba30c83013cf2f6efb9fcec8c3496be99b77ab0b800f97441648e9108215ab",
        "verified_on": "2026-10-10",
        "inspection": "actual_workbook"
      },
      "households": {
        "filename": "households_9_1.xlsx",
        "sheet": "b09_01",
        "first_data_row": 11,
        "stat_infid": "000032142495",
        "page_url": "https://www.e-stat.go.jp/stat-search/files?stat_infid=000032142495",
        "download_url": "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032142495&fileKind=0",
        "columns": {
          "region_type": "A",
          "prefecture": "B",
          "area_name_code": "C",
          "family_type": "E",
          "general_households": "F"
        },
        "header_anchors": {
          "A1": "令和２年国勢調査　人口等基本集計",
          "A2": "第９－１表　世帯の家族類型，世帯員の年齢による世帯の種類別一般世帯数及び3世代世帯数－全国，都道府県，市区町村",
          "A10": "地域識別コード",
          "B10": "都道府県",
          "C10": "地域名",
          "E10": "世帯の家族類型",
          "F5": "一般世帯数",
          "F8": "0_総数",
          "F9": "世帯"
        },
        "verified_file_sha256": "343c7a2797071cb63215bcaee640e2b43c6b212efb3b31dd379cd84206981f25",
        "verified_on": "2026-10-10",
        "inspection": "actual_workbook"
      },
      "age": {
        "filename": "age_2_7.xlsx",
        "sheet": "b02_07",
        "first_data_row": 13,
        "stat_infid": "000032142410",
        "page_url": "https://www.e-stat.go.jp/stat-search/files?stat_infid=000032142410",
        "download_url": "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000032142410&fileKind=0",
        "columns": {
          "nationality": "A",
          "sex": "B",
          "region_type": "C",
          "prefecture": "G",
          "area_code": "H",
          "area_name": "I",
          "age_total": "J",
          "age_unknown": "AF",
          "age_under15": "AG",
          "age_15_64": "AH",
          "age_65plus": "AI",
          "age_65plus_share_known_source_pct": "AO"
        },
        "header_anchors": {
          "A1": "令和２年国勢調査　人口等基本集計",
          "A2": "第２－７表　男女，年齢（5歳階級及び3区分），国籍総数か日本人別人口，平均年齢，年齢中位数及び人口構成比［年齢別］－全国，都道府県，市区町村（2000年（平成12年）市区町村含む）",
          "A4": "1) 「人口構成比［年齢別］」，「平均年齢」及び「年齢中位数」については，年齢「不詳」の者を除いて算出。",
          "A12": "国籍総数か日本人",
          "B12": "男女",
          "C12": "地域識別コード",
          "G12": "2020年_都道府県",
          "H12": "2020年_地域コード",
          "I12": "地域名",
          "J7": "人口",
          "J10": "00_総数",
          "AF10": "22_年齢「不詳」",
          "AG10": "R1_（再掲）15歳未満",
          "AH10": "R2_（再掲）15～64歳",
          "AI10": "R3_（再掲）65歳以上",
          "AO10": "3_65歳以上",
          "AF11": "人",
          "AG11": "人",
          "AI11": "人",
          "AO11": "％"
        },
        "verified_file_sha256": "114f2343dfaaa048614b0283830070b2048cae7da67a38df96c83039c6d0ac75",
        "verified_on": "2026-10-10",
        "inspection": "actual_workbook"
      }
    }
  },
  "2025": {
    "census_year": 2025,
    "reference_date": "2025-10-01",
    "population_previous_year": 2020,
    "series": "original",
    "boundary_basis": "survey_date",
    "unknown_policy": "preserve_dash_as_null",
    "sources": {
      "population": {
        "filename": "population_1_1.xlsx",
        "sheet": "b01_01",
        "first_data_row": 16,
        "stat_infid": "000040506423",
        "page_url": "https://www.e-stat.go.jp/stat-search/files?stat_infid=000040506423",
        "download_url": "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040506423&fileKind=0",
        "columns": {
          "region_type": "A",
          "prefecture": "E",
          "area_code": "F",
          "area_name": "G",
          "population": "H",
          "population_previous_rearranged": "K",
          "population_change_pct_source": "M",
          "households": "Q",
          "general_households": "R"
        },
        "header_anchors": {
          "A1": "【原数値】令和７年国勢調査　人口等基本集計",
          "A2": "第１－１表　男女別人口、世帯の種類別世帯数及び世帯人員並びに2020年（令和2年）の人口（組替）、2020年（令和2年）の世帯数（組替）、5年間の人口増減数、5年間の人口増減率、5年間の世帯増減数、5年間の世帯増減率、人口性比、面積（参考）及び人口密度－全国、都道府県、市区町村（2000年（平成12年）市区町村含む）",
          "A15": "地域識別コード",
          "E15": "2025年_都道府県",
          "F15": "2025年_地域コード",
          "G15": "地域名",
          "H10": "人口",
          "H12": "0_総数",
          "H14": "人",
          "K10": "2020年（令和2年）の人口（組替）",
          "K14": "人",
          "M10": "5年間の人口増減率",
          "M14": "％",
          "Q10": "世帯数",
          "R10": "世帯数",
          "R12": "1_一般世帯",
          "R14": "世帯"
        },
        "verified_file_sha256": "c1ff781e05af6fce6510ee9cfbeba856106c97ae780ca030a549466b0074d856",
        "verified_on": "2026-10-10",
        "inspection": "actual_workbook"
      },
      "households": {
        "filename": "households_9_1.xlsx",
        "sheet": "b09_01",
        "first_data_row": 11,
        "stat_infid": "000040506456",
        "page_url": "https://www.e-stat.go.jp/stat-search/files?stat_infid=000040506456",
        "download_url": "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040506456&fileKind=0",
        "columns": {
          "region_type": "A",
          "prefecture": "B",
          "area_name_code": "C",
          "family_type": "E",
          "general_households": "F"
        },
        "header_anchors": {
          "A1": "【原数値】令和７年国勢調査　人口等基本集計",
          "A2": "第９－１表　世帯の家族類型、世帯員の年齢による世帯の種類別一般世帯数及び3世代世帯数－全国、都道府県、市区町村",
          "A10": "地域識別コード",
          "B10": "都道府県",
          "C10": "地域名",
          "E10": "世帯の家族類型",
          "F5": "一般世帯数",
          "F7": "0_総数",
          "F9": "世帯"
        },
        "verified_file_sha256": "e0d5153cbad49adcca91a0aac410118dd6445deb9d47972923b82d2c10011984",
        "verified_on": "2026-10-10",
        "inspection": "actual_workbook"
      },
      "age": {
        "filename": "age_2_7.xlsx",
        "sheet": "b02_07",
        "first_data_row": 13,
        "stat_infid": "000040506431",
        "page_url": "https://www.e-stat.go.jp/stat-search/files?stat_infid=000040506431",
        "download_url": "https://www.e-stat.go.jp/stat-search/file-download?statInfId=000040506431&fileKind=0",
        "columns": {
          "nationality": "A",
          "sex": "B",
          "region_type": "C",
          "prefecture": "G",
          "area_code": "H",
          "area_name": "I",
          "age_total": "J",
          "age_unknown": "AF",
          "age_under15": "AG",
          "age_15_64": "AH",
          "age_65plus": "AI",
          "age_65plus_share_known_source_pct": "AO"
        },
        "header_anchors": {
          "A1": "【原数値】令和７年国勢調査　人口等基本集計",
          "A2": "第２－７表　男女、年齢（5歳階級及び3区分）、国籍総数か日本人別人口、平均年齢、年齢中位数及び人口構成比［年齢別］－全国、都道府県、市区町村（2000年（平成12年）市区町村含む）",
          "A4": "1) 「人口構成比［年齢別］」、「平均年齢」及び「年齢中位数」については、年齢「不詳」の者を除いて算出。",
          "A12": "国籍総数か日本人",
          "B12": "男女",
          "C12": "地域識別コード",
          "G12": "2025年_都道府県",
          "H12": "2025年_地域コード",
          "I12": "地域名",
          "J7": "人口",
          "J9": "00_総数",
          "AF9": "22_年齢「不詳」",
          "AG9": "R1_（再掲）15歳未満",
          "AH9": "R2_（再掲）15～64歳",
          "AI9": "R3_（再掲）65歳以上",
          "AO9": "3_65歳以上",
          "AF11": "人",
          "AG11": "人",
          "AI11": "人",
          "AO11": "％"
        },
        "verified_file_sha256": "3d0dab20893d8fca43ef33d521a0651841de105cf5b2720582e5024469030e75",
        "verified_on": "2026-10-10",
        "inspection": "actual_workbook"
      }
    }
  }
}
''')
CONFIG = SOURCE_REGISTRY[str(CENSUS_YEAR)]


## 共通処理と年別の列定義

旧市町村行・特別区部合計を除外し、現行62コードを選びます。各表の名前・人口・一般世帯数、都総数との一致も検査します。


In [ ]:
"""Original census Excel adapters. Used verbatim inside nb01; no Spark needed here."""
from pathlib import Path
from decimal import Decimal, InvalidOperation
import hashlib
import json
import math
import re
import openpyxl
from openpyxl.utils.cell import column_index_from_string, coordinate_to_tuple

TOKYO_CODES = set(
    [str(x) for x in range(13101, 13124)]
    + [str(x) for x in range(13201, 13216)]
    + [str(x) for x in range(13218, 13226)]
    + ['13227', '13228', '13229', '13303', '13305', '13307', '13308',
       '13361', '13362', '13363', '13364', '13381', '13382', '13401', '13402', '13421']
)
assert len(TOKYO_CODES) == 62


def count_value(value, field, nullable_dash=False):
    """Keep '-' only for age_unknown; never silently coerce unexpected symbols."""
    if nullable_dash and value == '-':
        return None
    if value is None or isinstance(value, bool):
        raise ValueError(f'{field}: empty/non-count value {value!r}')
    try:
        number = Decimal(str(value).replace(',', '').strip())
    except InvalidOperation as exc:
        raise ValueError(f'{field}: unexpected count symbol {value!r}') from exc
    if not number.is_finite() or number != number.to_integral_value() or number < 0:
        raise ValueError(f'{field}: invalid nonnegative integer {value!r}')
    return int(number)


def decimal_value(value, field):
    try:
        number = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'{field}: unexpected percentage {value!r}') from exc
    if not math.isfinite(number):
        raise ValueError(f'{field}: non-finite percentage')
    return number


def clean_name(value):
    value = str(value)
    if '_' not in value:
        raise ValueError(f'Area-name code prefix missing: {value!r}')
    return value.split('_', 1)[1]


def canonical_hash(records):
    """Fingerprint canonical curated values; identical to comparison_core.canonical_hash."""
    return hashlib.sha256(json.dumps(sorted(records, key=lambda r: r['area_code']),
        ensure_ascii=False, sort_keys=True, allow_nan=False, separators=(',', ':')).encode('utf-8')).hexdigest()


def extract_year(raw_year_dir, config):
    """Return (62 canonical records, verification details) using year-specific anchors."""
    population, families, ages, source_files, aggregate = {}, {}, {}, {}, {}
    for kind, definition in config['sources'].items():
        source_path = Path(raw_year_dir) / definition['filename']
        if not source_path.is_file():
            raise FileNotFoundError(f'Upload the official original workbook: {source_path}')
        digest = hashlib.sha256(source_path.read_bytes()).hexdigest()
        workbook = openpyxl.load_workbook(source_path, read_only=True, data_only=True)
        try:
            if definition['sheet'] not in workbook.sheetnames:
                raise ValueError(f'{kind}: expected sheet {definition["sheet"]!r} not found')
            sheet = workbook[definition['sheet']]
            header = list(sheet.iter_rows(max_row=definition['first_data_row']-1, values_only=True))
            for address, expected in definition['header_anchors'].items():
                rr, cc = coordinate_to_tuple(address)
                actual = header[rr-1][cc-1]
                if actual != expected:
                    raise ValueError(f'{kind} {address}: expected {expected!r}; actual {actual!r}')
            # Check the actual headings before the file fingerprint so structural drift is explicit.
            if digest != definition['verified_file_sha256']:
                raise ValueError(f'{kind}: SHA-256 differs from the inspected source. Check official '
                                 'revision information, re-inspect totals, then revise the registry.')
            columns = {key: column_index_from_string(letter)-1
                       for key, letter in definition['columns'].items()}
            source_files[kind] = {'path': str(source_path), 'sha256': digest,
                                 'stat_infid': definition['stat_infid'],
                                 'page_url': definition['page_url'], 'header_check': 'PASS'}
            for source_row, values in enumerate(sheet.iter_rows(
                    min_row=definition['first_data_row'], values_only=True),
                    definition['first_data_row']):
                def value(field):
                    return values[columns[field]]
                if value('prefecture') != '13_東京都':
                    continue
                if kind == 'households':
                    code = str(value('area_name_code')).split('_', 1)[0]
                else:
                    code = str(value('area_code'))
                is_total = code == '13000' and value('region_type') == 'a'
                is_municipality = code in TOKYO_CODES and str(value('region_type')) in {'0', '2', '3'}
                if not (is_total or is_municipality):
                    continue  # Tokyo/ward aggregate and historical boundary rows are excluded.
                if kind == 'population':
                    if code in population:
                        raise ValueError(f'population duplicate code: {code}')
                    population[code] = {'area_code': code, 'area_name': clean_name(value('area_name')),
                        'population': count_value(value('population'), 'population'),
                        'general_households': count_value(value('general_households'), 'general_households'),
                        'households': count_value(value('households'), 'households'),
                        'population_previous_rearranged': count_value(value('population_previous_rearranged'), 'population_previous_rearranged'),
                        'population_change_pct_source': decimal_value(value('population_change_pct_source'), 'population_change_pct_source'),
                        'population_source_row': source_row}
                elif kind == 'households':
                    family_type = value('family_type')
                    if family_type not in {'0_総数', '3_単独世帯'}:
                        continue
                    family_key = 'general_households' if family_type == '0_総数' else 'single_households'
                    entry = families.setdefault(code, {'area_name': clean_name(value('area_name_code'))})
                    if family_key in entry:
                        raise ValueError(f'households duplicate: {code}/{family_type}')
                    entry[family_key] = count_value(value('general_households'), family_key)
                    entry[family_key+'_source_row'] = source_row
                elif kind == 'age':
                    if value('nationality') != '0_国籍総数' or value('sex') != '0_総数':
                        continue
                    if code in ages:
                        raise ValueError(f'age duplicate code: {code}')
                    ages[code] = {'area_name': clean_name(value('area_name')),
                        **{field: count_value(value(field), field, nullable_dash=field=='age_unknown')
                           for field in ['age_total', 'age_under15', 'age_15_64', 'age_65plus', 'age_unknown']},
                        'age_unknown_raw': str(value('age_unknown')),
                        'age_65plus_share_known_source_pct': decimal_value(value('age_65plus_share_known_source_pct'), 'age_65plus_share_known_source_pct'),
                        'age_source_row': source_row}
        finally:
            workbook.close()

    expected_keys = TOKYO_CODES | {'13000'}
    for name, rows in [('population', population), ('households', families), ('age', ages)]:
        if set(rows) != expected_keys:
            raise ValueError(f'{name}: unexpected area set; missing={sorted(expected_keys-set(rows))}; '
                             f'extra={sorted(set(rows)-expected_keys)}')
    output = []
    for code in sorted(expected_keys):
        p, h, a = population[code], families[code], ages[code]
        if not p['area_name'] == h['area_name'] == a['area_name']:
            raise ValueError(f'{code}: area names differ between tables')
        if h.get('general_households') != p['general_households'] or 'single_households' not in h:
            raise ValueError(f'{code}: general households differ between tables / single-household row missing')
        if a['age_total'] != p['population']:
            raise ValueError(f'{code}: age total and population differ')
        known = sum(a[f] for f in ['age_under15', 'age_15_64', 'age_65plus'])
        if min(p['population'], p['population_previous_rearranged'], p['general_households'], known) <= 0:
            raise ValueError(f'{code}: nonpositive denominator')
        if not 0 <= h['single_households'] <= p['general_households'] <= p['households']:
            raise ValueError(f'{code}: invalid household counts')
        if known > p['population']:
            raise ValueError(f'{code}: known-age population exceeds total population')
        if a['age_unknown'] is not None and known+a['age_unknown'] != p['population']:
            raise ValueError(f'{code}: age groups do not sum to total')
        if abs(a['age_65plus']/known*100-a['age_65plus_share_known_source_pct']) > 0.000011:
            raise ValueError(f'{code}: recalculated elderly share disagrees with source')
        if abs((p['population']/p['population_previous_rearranged']-1)*100-p['population_change_pct_source']) > 0.000011:
            raise ValueError(f'{code}: recalculated population growth disagrees with source')
        record = {**p, **{f: a[f] for f in ['age_under15', 'age_15_64', 'age_65plus', 'age_unknown',
                                          'age_unknown_raw', 'age_65plus_share_known_source_pct', 'age_source_row']},
                  'single_households': h['single_households'],
                  'single_source_row': h['single_households_source_row'],
                  'census_year': config['census_year'], 'reference_date': config['reference_date'],
                  'population_previous_year': config['population_previous_year'], 'series': 'original',
                  'boundary_basis': config['boundary_basis'], 'unknown_policy': config['unknown_policy']}
        if code == '13000':
            aggregate = record
        else:
            output.append(record)
    # Exact additive checks against the Tokyo aggregate, independent of the 62 row filter.
    fields = ['population', 'general_households', 'households', 'single_households',
              'age_under15', 'age_15_64', 'age_65plus', 'population_previous_rearranged']
    totals = {field: sum(row[field] for row in output) for field in fields}
    for field in fields:
        if totals[field] != aggregate[field]:
            raise ValueError(f'Tokyo aggregate mismatch for {field}: {totals[field]} != {aggregate[field]}')
    unknown_null_codes = [row['area_code'] for row in output if row['age_unknown'] is None]
    verification = {'status': 'PASS', 'census_year': config['census_year'], 'rows': len(output),
        'reference_date': config['reference_date'], 'series': 'original',
        'boundary_basis': config['boundary_basis'], 'unknown_policy': config['unknown_policy'],
        'totals': totals, 'tokyo_aggregate': aggregate,
        'curated_sha256': canonical_hash(output),
        'age_unknown_nulls': len(unknown_null_codes), 'age_unknown_null_codes': unknown_null_codes,
        'source_files': source_files,
        'note': 'Original dash symbols remain NULL. Unknown-rate deltas remain NULL if either year is NULL.'}
    return output, verification


In [ ]:
records, source_validation = extract_year(f'{RAW_ROOT}/{CENSUS_YEAR}', CONFIG)
print(json.dumps({
    'status': source_validation['status'],
    'census_year': CENSUS_YEAR,
    'rows': source_validation['rows'],
    'totals': source_validation['totals'],
    'age_unknown_null_codes': source_validation['age_unknown_null_codes'],
}, ensure_ascii=False, indent=2))


## Sparkの共通スキーマにそろえて保存する

既存runは上書きしません。途中で失敗した場合も新しいsource_run_idで再実行します。
PASSはParquetの保存後、読戻しと内容一致を確認した最後に書き込みます。


In [ ]:
string_fields = {
    'area_code', 'area_name', 'reference_date', 'series',
    'boundary_basis', 'unknown_policy', 'age_unknown_raw'
}
double_fields = {'population_change_pct_source', 'age_65plus_share_known_source_pct'}
columns = list(records[0].keys())
schema = T.StructType([
    T.StructField(name, T.StringType() if name in string_fields else
                  T.DoubleType() if name in double_fields else T.LongType(),
                  nullable=(name == 'age_unknown'))
    for name in columns
])
year_df = spark.createDataFrame(records, schema=schema)
if year_df.count() != 62 or year_df.select('area_code').distinct().count() != 62:
    raise ValueError('Expected 62 unique municipalities')
if {row.area_code for row in year_df.select('area_code').collect()} != TOKYO_CODES:
    raise ValueError('Municipality-code mismatch')
required = [f.name for f in schema.fields if not f.nullable]
if year_df.filter(F.expr(' OR '.join(f'`{c}` IS NULL' for c in required))).limit(1).count():
    raise ValueError('Unexpected NULL in required columns')
spark_totals = year_df.agg(*[F.sum(c).alias(c) for c in source_validation['totals']]).first().asDict()
if spark_totals != source_validation['totals']:
    raise ValueError('Spark totals differ from validated source totals')

Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
year_df.write.mode('errorifexists').parquet(TARGET)
roundtrip = [row.asDict() for row in spark.read.parquet(TARGET).orderBy('area_code').collect()]
if roundtrip != sorted(records, key=lambda row: row['area_code']):
    raise ValueError('Parquet readback changed values')
validation = {**source_validation, 'source_run_id': SOURCE_RUN_ID,
              'target': TARGET, 'spark_version': spark.version,
              'parquet_readback': 'PASS'}
with (Path(RUN_ROOT)/'validation.json').open('x', encoding='utf-8') as out:
    json.dump(validation, out, ensure_ascii=False, indent=2, allow_nan=False)
print(json.dumps({'status': 'PASS', 'rows': 62, 'census_year': CENSUS_YEAR,
                  'source_run_id': SOURCE_RUN_ID, 'target': TARGET}, ensure_ascii=False))
year_df.orderBy('area_code').show(62, truncate=False)
